In [1]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes


In [1]:
import torch

print("Device:", torch.cuda.get_device_name(0))
print("Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


Device: Tesla T4
Memory (GB): 15.64


In [4]:
import os
from google.colab import files

uploaded = files.upload()

expected = ["train.jsonl", "dev.jsonl", "test_inputs.jsonl", "score.py", "SCHEMA.md"]
for f in expected:
    print(f, "-> FOUND" if os.path.exists(f) else "-> MISSING!")


Saving dev.jsonl to dev.jsonl
Saving test_inputs.jsonl to test_inputs.jsonl
Saving train.jsonl to train.jsonl
train.jsonl -> FOUND
dev.jsonl -> FOUND
test_inputs.jsonl -> FOUND
score.py -> MISSING!
SCHEMA.md -> MISSING!


In [5]:
import os
from google.colab import files

uploaded = files.upload()

expected = ["train.jsonl", "dev.jsonl", "test_inputs.jsonl", "score.py", "SCHEMA.md"]
for f in expected:
    print(f, "-> FOUND" if os.path.exists(f) else "-> MISSING!")


Saving SCHEMA.md to SCHEMA.md
Saving score.py to score.py
train.jsonl -> FOUND
dev.jsonl -> FOUND
test_inputs.jsonl -> FOUND
score.py -> FOUND
SCHEMA.md -> FOUND


In [13]:
import json, re, csv

with open("SCHEMA.md", "r", encoding="utf-8") as f:
    print("=== SCHEMA.md PREVIEW ===")
    print(f.read()[:1500])

raw_train = load_jsonl("train.jsonl")
clean_train = []
log_rows = []
seen_tickets = set()


valid_cats = {"account_access", "fraud", "kyc", "offers", "other", "payment_failed", "refund"}
no_txn_cats = {"account_access", "kyc", "other"}
valid_channels = {"upi", "card", "wallet", "netbanking", "unknown"}
old_prio_map = {"high": "P1", "medium": "P2", "low": "P3"}

for row in raw_train:
    row_id = row["id"]
    ticket = next(m["content"] for m in row["messages"] if m["role"] == "user")
    raw_ans = row["messages"][-1]["content"]
    fixes = []

    try:
        ans = json.loads(raw_ans)
    except Exception:
        log_rows.append([row_id, "DROPPED", "Invalid or cut-off JSON"])
        continue

    body = " ".join(ln for ln in ticket.splitlines() if ":" not in ln[:15]).strip().lower()
    if len(body) < 8 or re.fullmatch(r"(test|asdf|qwerty|hello|hi|dummy|123|\W|\s)+", body):
        log_rows.append([row_id, "DROPPED", "Junk or test ticket"])
        continue

    norm_ticket = " ".join(ticket.lower().split())
    if norm_ticket in seen_tickets:
        log_rows.append([row_id, "DROPPED", "Duplicate ticket"])
        continue
    seen_tickets.add(norm_ticket)

    if "prio" in ans:
        ans["priority"] = old_prio_map.get(str(ans.pop("prio")).lower(), "P3")
        fixes.append("prio -> priority")

    cat = str(ans.get("category", "other")).lower().strip()
    if cat not in valid_cats:
        cat = "other"
        fixes.append("fixed category")
    ans["category"] = cat

    amt = ans.get("amount_paise")
    if isinstance(amt, (int, float)) and amt > 0:
        amt = round(amt)
        rupee_nums = [round(float(x.replace(",", ""))) for x in re.findall(r"(?:₹|rs\.?|inr)\s*([\d,]+(?:\.\d+)?)", ticket, re.I)]
        if amt in rupee_nums and (amt * 100) not in rupee_nums:
            amt = amt * 100
            fixes.append("rupees -> paise")
        ans["amount_paise"] = amt
    else:
        ans["amount_paise"] = None

    dt = ans.get("txn_date")
    if isinstance(dt, str) and re.fullmatch(r"\d{2}/\d{2}/\d{4}", dt.strip()):
        d, m, y = dt.strip().split("/")
        ans["txn_date"] = f"{y}-{m}-{d}"
        fixes.append("date format")
    elif not (isinstance(dt, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", dt.strip())):
        ans["txn_date"] = None

    tid = ans.get("txn_id")
    if isinstance(tid, str):
        clean_tid = re.sub(r"[\s\-]", "", tid).upper()
        ans["txn_id"] = clean_tid if re.fullmatch(r"NP\d{10}", clean_tid) else None
    else:
        ans["txn_id"] = None

    ch = str(ans.get("channel") or "unknown").lower().replace("-", "").strip()
    ans["channel"] = ch if ch in valid_channels else "unknown"

    if cat in no_txn_cats:
        ans["amount_paise"] = None
        ans["txn_date"] = None
        ans["txn_id"] = None
        ans["channel"] = None


    lang = str(ans.get("language") or "en").lower().strip()
    if lang not in {"en", "hi", "hinglish"}:
        lang = "en"

    low_t = ticket.lower()
    p1_words = ["fraud", "scam", "unauthorized", "unauthorised", "hacked", "rbi", "ombudsman", "consumer court", "police"]
    if cat == "fraud" or any(w in low_t for w in p1_words) or (ans["amount_paise"] and ans["amount_paise"] >= 5000000):
        ans["priority"] = "P1"
        ans["needs_human"] = True
    elif ans.get("priority") not in {"P1", "P2", "P3"}:
        ans["priority"] = "P3"
    ans["needs_human"] = True if ans["priority"] == "P1" else bool(ans.get("needs_human", False))

    clean_json = json.dumps({
        "category": ans["category"],
        "priority": ans["priority"],
        "amount_paise": ans["amount_paise"],
        "txn_date": ans["txn_date"],
        "txn_id": ans["txn_id"],
        "channel": ans["channel"],
        "language": lang,
        "needs_human": ans["needs_human"],
    }, ensure_ascii=False)

    clean_train.append({
        "id": row_id,
        "messages": [row["messages"][0], row["messages"][1], {"role": "assistant", "content": clean_json}]
    })
    if fixes:
        log_rows.append([row_id, "FIXED", ", ".join(fixes)])

save_jsonl("train_clean.jsonl", clean_train)
with open("cleaning_log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "action", "reason"])
    w.writerows(log_rows)

print(f"\nUpdated train_clean.jsonl: {len(clean_train)} rows | Logged actions: {len(log_rows)}")


=== SCHEMA.md PREVIEW ===
# NimbusPay Triage Labelling Spec (v2, current)

NimbusPay is a fictional payments app. Every incoming support ticket is turned into one JSON object, the *triage record*. This document is the single source of truth for what a correct record looks like. **All dev and test labels follow this spec exactly.**

## Input

```
Received: 2026-08-14 (Fri)
Source: email

<the customer's message>
```

`Received` is the date the ticket arrived. `Source` is one of `app-chat`, `email`, `twitter`, `ivr-transcript`.

## Output

A single JSON object with **exactly these 8 keys**. Use `null` when a value is not present in the ticket.

```json
{"category": "refund", "priority": "P2", "amount_paise": 2936300, "txn_date": "2026-09-13", "txn_id": "NP5972758064", "channel": "netbanking", "language": "en", "needs_human": false}
```

| Key | Type | Allowed values |
|---|---|---|
| `category` | string | `payment_failed`, `refund`, `fraud`, `account_access`, `kyc`, `offers`, `other` |
|

In [15]:
import gc
import torch
import subprocess
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MAX_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

dev_rows = load_jsonl("dev.jsonl")
test_rows = load_jsonl("test_inputs.jsonl")

def predict_all(mdl, tok, rows, out_file, custom_sys=None, batch_size=4):
    mdl.generation_config.max_length = None
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id

    prompts = []
    for r in rows:
        user_msg = next(m["content"] for m in r["messages"] if m["role"] == "user")
        sys_msg = custom_sys if custom_sys else next(m["content"] for m in r["messages"] if m["role"] == "system")
        msgs = [{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}]
        prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

    preds = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i + batch_size]
        inputs = tok(batch_prompts, return_tensors="pt", padding=True).to("cuda")
        input_len = inputs["input_ids"].size(1)
        with torch.no_grad():
            out = mdl.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
        for j, seq in enumerate(out):
            reply = tok.decode(seq[input_len:], skip_special_tokens=True).strip()
            preds.append({"id": rows[i + j]["id"], "output": reply})

    save_jsonl(out_file, preds)
    return preds

# Baseline 1: Default short prompt
predict_all(model, tokenizer, dev_rows, "base_short.jsonl", batch_size=4)
print("--- BASELINE 1: SHORT PROMPT ---")
print(subprocess.check_output(["python", "score.py", "--gold", "dev.jsonl", "--pred", "base_short.jsonl"]).decode())

# Baseline 2: Full SCHEMA.md rules in prompt
with open("SCHEMA.md", "r", encoding="utf-8") as f:
    rules_text = "Return ONLY valid JSON following these rules:\n\n" + f.read()

predict_all(model, tokenizer, dev_rows, "base_rules.jsonl", custom_sys=rules_text, batch_size=4)
print("--- BASELINE 2: FULL RULES IN PROMPT ---")
print(subprocess.check_output(["python", "score.py", "--gold", "dev.jsonl", "--pred", "base_rules.jsonl"]).decode())


del model
gc.collect()
torch.cuda.empty_cache()


==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

--- BASELINE 1: SHORT PROMPT ---
group                            n   valid   exact  fields
all                            200   87.5%    0.0%    0.0%
category=account_access         19   94.7%    0.0%    0.0%
category=fraud                  15   93.3%    0.0%    0.0%
category=kyc                    24   91.7%    0.0%    0.0%
category=offers                 18   94.4%    0.0%    0.0%
category=other                  16   93.8%    0.0%    0.0%
category=payment_failed         53   86.8%    0.0%    0.0%
category=refund                 55   78.2%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%

--- BASELINE 2: FULL RULES IN PROMPT ---
group                            n   valid   exact  fields
all                            200    2.5%    0.0%    1.8%
category=account_access         19    5.3%    0.0%   

In [18]:
import json, re, csv

raw_train = load_jsonl("train.jsonl")
dev_rows = load_jsonl("dev.jsonl")

print("Sample DEV gold:", dev_rows[0]["messages"][-1]["content"])
print("Sample TRAIN gold:", raw_train[0]["messages"][-1]["content"])

clean_train = []
log_rows = []
seen_tickets = set()

valid_cats = {"account_access", "fraud", "kyc", "offers", "other", "payment_failed", "refund"}
no_txn_cats = {"account_access", "kyc", "other"}
valid_channels = {"upi", "card", "wallet", "netbanking", "unknown"}
old_prio_map = {"high": "P1", "medium": "P2", "low": "P3"}

for row in raw_train:
    row_id = row["id"]
    sys_msg = next(m["content"] for m in row["messages"] if m["role"] == "system")
    ticket = next(m["content"] for m in row["messages"] if m["role"] == "user")
    raw_ans = row["messages"][-1]["content"]
    fixes = []

    try:
        ans = json.loads(raw_ans)
        if not isinstance(ans, dict):
            raise ValueError()
    except Exception:
        log_rows.append([row_id, "DROPPED", "Invalid or cut-off JSON"])
        continue

    body = " ".join(ln for ln in ticket.splitlines() if ":" not in ln[:15]).strip().lower()
    if len(body) < 8 or re.fullmatch(r"(test|asdf|qwerty|hello|hi|dummy|123|\W|\s)+", body):
        log_rows.append([row_id, "DROPPED", "Junk or test ticket"])
        continue

    norm_ticket = " ".join(ticket.lower().split())
    if norm_ticket in seen_tickets:
        log_rows.append([row_id, "DROPPED", "Duplicate ticket"])
        continue
    seen_tickets.add(norm_ticket)

    if "prio" in ans:
        ans["priority"] = old_prio_map.get(str(ans.pop("prio")).lower(), "P3")
        fixes.append("prio -> priority")

    cat = str(ans.get("category", "other")).lower().strip()
    if cat not in valid_cats:
        cat = "other"
        fixes.append("fixed category")
    ans["category"] = cat

    amt = ans.get("amount_paise")
    if isinstance(amt, (int, float)) and amt > 0:
        amt = round(amt)
        rupee_nums = [round(float(x.replace(",", ""))) for x in re.findall(r"(?:₹|rs\.?|inr)\s*([\d,]+(?:\.\d+)?)", ticket, re.I)]
        if amt in rupee_nums and (amt * 100) not in rupee_nums:
            amt = amt * 100
            fixes.append("rupees -> paise")
        ans["amount_paise"] = amt
    else:
        ans["amount_paise"] = None

    dt = ans.get("txn_date")
    if isinstance(dt, str) and re.fullmatch(r"\d{2}/\d{2}/\d{4}", dt.strip()):
        d, m, y = dt.strip().split("/")
        ans["txn_date"] = f"{y}-{m}-{d}"
        fixes.append("date format")
    elif not (isinstance(dt, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", dt.strip())):
        ans["txn_date"] = None

    tid = ans.get("txn_id")
    if isinstance(tid, str):
        clean_tid = re.sub(r"[\s\-]", "", tid).upper()
        ans["txn_id"] = clean_tid if re.fullmatch(r"NP\d{10}", clean_tid) else None
    else:
        ans["txn_id"] = None

    ch = str(ans.get("channel") or "unknown").lower().replace("-", "").strip()
    ans["channel"] = ch if ch in valid_channels else "unknown"

    if cat in no_txn_cats:
        ans["amount_paise"] = None
        ans["txn_date"] = None
        ans["txn_id"] = None
        ans["channel"] = None

    lang = str(ans.get("language") or "en").lower().strip()
    if lang not in {"en", "hi", "hinglish"}:
        lang = "en"

    low_t = ticket.lower()
    p1_words = ["fraud", "scam", "unauthorized", "unauthorised", "hacked", "rbi", "ombudsman", "consumer court", "police"]
    if cat == "fraud" or any(w in low_t for w in p1_words) or (ans["amount_paise"] and ans["amount_paise"] >= 5000000):
        ans["priority"] = "P1"
        ans["needs_human"] = True
    elif ans.get("priority") not in {"P1", "P2", "P3"}:
        ans["priority"] = "P3"
    ans["needs_human"] = True if ans["priority"] == "P1" else bool(ans.get("needs_human", False))

    clean_json = json.dumps({
        "category": ans["category"],
        "priority": ans["priority"],
        "amount_paise": ans["amount_paise"],
        "txn_date": ans["txn_date"],
        "txn_id": ans["txn_id"],
        "channel": ans["channel"],
        "language": lang,
        "needs_human": ans["needs_human"],
    }, ensure_ascii=False)

    clean_train.append({
        "id": row_id,
        "messages": [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": ticket},
            {"role": "assistant", "content": clean_json}
        ]
    })
    if fixes:
        log_rows.append([row_id, "FIXED", ", ".join(fixes)])

save_jsonl("train_clean.jsonl", clean_train)
with open("cleaning_log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "action", "reason"])
    w.writerows(log_rows)

print(f"Saved {len(clean_train)} clean rows | Logged {len(log_rows)} actions.")
print("Sample CLEANED gold:", clean_train[0]["messages"][-1]["content"])



Sample DEV gold: {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}
Sample TRAIN gold: {"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}
Saved 1333 clean rows | Logged 195 actions.
Sample CLEANED gold: {"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}


In [19]:
results = [
    {"name": "exp1_raw", "exact_match": 95.0, "field_acc": 99.1, "gpu_gb": 8.38, "mins": 23.96},
    train_model("exp2_clean", clean_train, use_4bit=True, rank=16, lr=2e-4, epochs=3),
    train_model("exp3_16bit", clean_train, use_4bit=False, rank=16, lr=2e-4, epochs=3),
    train_model("exp4_rank32", clean_train, use_4bit=True, rank=32, lr=2e-4, epochs=3),
    train_model("exp5_4ep_lr1e4", clean_train, use_4bit=True, rank=16, lr=1e-4, epochs=4),
]

print("\n=== ALL EXPERIMENT RESULTS ===")
for r in results:
    print(r)


==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1333 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/1333 [00:00<?, ? examples/s]

... (8 outputs left out)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,333 | Num Epochs = 3 | Total steps = 252
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
30,0.532053


KeyboardInterrupt: 

In [20]:
import subprocess

print("=== WHY exp2_clean SCORED 57.0% ===")
print(subprocess.check_output(["python", "score.py", "--gold", "dev.jsonl", "--pred", "pred_exp2_clean.jsonl"]).decode())

with open("SCHEMA.md", "r", encoding="utf-8") as f:
    print("=== FULL SCHEMA.md ===")
    print(f.read())


=== WHY exp2_clean SCORED 57.0% ===
group                            n   valid   exact  fields
all                            200  100.0%   57.0%   92.6%
category=account_access         19  100.0%   73.7%   96.1%
category=fraud                  15  100.0%   46.7%   87.5%
category=kyc                    24  100.0%   83.3%   97.9%
category=offers                 18  100.0%   16.7%   88.9%
category=other                  16  100.0%   75.0%   96.9%
category=payment_failed         53  100.0%   58.5%   93.2%
category=refund                 55  100.0%   49.1%   90.0%

per-field accuracy (all rows)
  category        99.0%
  priority        95.0%
  amount_paise    99.5%
  txn_date        99.5%
  txn_id          99.5%
  channel         76.0%
  language        75.5%
  needs_human     97.0%

=== FULL SCHEMA.md ===
# NimbusPay Triage Labelling Spec (v2, current)

NimbusPay is a fictional payments app. Every incoming support ticket is turned into one JSON object, the *triage record*. This document i

In [21]:
import json, re, csv

raw_train = load_jsonl("train.jsonl")
clean_train = []
log_rows = []
seen_tickets = set()

valid_cats = {"payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"}
no_txn_cats = {"account_access", "kyc", "other"}
valid_channels = {"upi", "card", "netbanking", "wallet"}
old_prio_map = {"high": "P1", "medium": "P2", "low": "P3"}

for row in raw_train:
    row_id = row["id"]
    sys_msg = next(m["content"] for m in row["messages"] if m["role"] == "system")
    ticket = next(m["content"] for m in row["messages"] if m["role"] == "user")
    raw_ans = row["messages"][-1]["content"]
    fixes = []

    try:
        ans = json.loads(raw_ans)
        if not isinstance(ans, dict):
            raise ValueError()
    except Exception:
        log_rows.append([row_id, "DROPPED", "Invalid or truncated JSON"])
        continue


    body = " ".join(ln for ln in ticket.splitlines() if not ln.startswith(("Received:", "Source:"))).strip().lower()
    if len(body) < 8 or re.fullmatch(r"(test|asdf|qwerty|hello|hi|dummy|123|\W|\s)+", body):
        log_rows.append([row_id, "DROPPED", "Empty or placeholder ticket"])
        continue


    norm_ticket = " ".join(ticket.lower().split())
    if norm_ticket in seen_tickets:
        log_rows.append([row_id, "DROPPED", "Duplicate ticket text"])
        continue
    seen_tickets.add(norm_ticket)


    if "prio" in ans:
        ans["priority"] = old_prio_map.get(str(ans.pop("prio")).lower().strip(), "P3")
        fixes.append("Mapped legacy prio -> priority")


    cat = str(ans.get("category", "other")).lower().strip()
    if cat not in valid_cats:
        cat = "other"
        fixes.append("Normalized invalid category -> other")
    ans["category"] = cat


    amt = ans.get("amount_paise")
    if isinstance(amt, (int, float)) and amt > 0:
        amt = round(amt)
        rupee_nums = [round(float(x.replace(",", ""))) for x in re.findall(r"(?:₹|rs\.?|inr)\s*([\d,]+(?:\.\d+)?)", ticket, re.I)]
        if amt in rupee_nums and (amt * 100) not in rupee_nums:
            amt = amt * 100
            fixes.append("Converted rupees -> paise (x100)")
        ans["amount_paise"] = amt
    else:
        ans["amount_paise"] = None

    dt = ans.get("txn_date")
    if isinstance(dt, str) and re.fullmatch(r"\d{2}/\d{2}/\d{4}", dt.strip()):
        d, m, y = dt.strip().split("/")
        ans["txn_date"] = f"{y}-{m}-{d}"
        fixes.append("Normalized txn_date DD/MM/YYYY -> YYYY-MM-DD")
    elif not (isinstance(dt, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", dt.strip())):
        ans["txn_date"] = None

    tid = ans.get("txn_id")
    if isinstance(tid, str):
        clean_tid = re.sub(r"[\s\-]", "", tid).upper()
        if re.fullmatch(r"NP\d{10}", clean_tid):
            if clean_tid != tid:
                fixes.append("Normalized txn_id format")
            ans["txn_id"] = clean_tid
        else:
            ans["txn_id"] = None
            fixes.append("Removed non-NP txn_id")
    else:
        ans["txn_id"] = None


    ch = ans.get("channel")
    if isinstance(ch, str):
        ch_clean = ch.lower().replace("-", "").replace(" ", "").strip()
        if ch_clean in {"internetbanking"}:
            ch_clean = "netbanking"
        ans["channel"] = ch_clean if ch_clean in valid_channels else None
        if ans["channel"] != ch:
            fixes.append("Normalized channel to schema allowed values/null")
    else:
        ans["channel"] = None

    if cat in no_txn_cats:
        for f_name in ["amount_paise", "txn_date", "txn_id", "channel"]:
            if ans.get(f_name) is not None:
                ans[f_name] = None
                fixes.append(f"Nulled {f_name} for {cat}")

    lang = str(ans.get("language") or "en").lower().strip()
    if lang in {"hinglish", "hi", "hi_en"}:
        lang = "hi-en"
        fixes.append("Normalized language -> hi-en")
    elif lang not in {"en", "hi-en"}:
        lang = "en"


    low_t = ticket.lower()
    legal_words = ["rbi", "ombudsman", "consumer court", "lawyer", "legal notice", "legal action"]
    if cat == "fraud" or (ans["amount_paise"] and ans["amount_paise"] >= 5_000_000) or any(w in low_t for w in legal_words):
        expected_prio = "P1"
    elif (cat in {"payment_failed", "refund"} and ans["amount_paise"] and ans["amount_paise"] >= 500_000) or cat == "account_access":
        expected_prio = "P2"
    else:
        expected_prio = "P3"

    if ans.get("priority") != expected_prio:
        fixes.append(f"Fixed priority {ans.get('priority')} -> {expected_prio}")
        ans["priority"] = expected_prio


    repeat_words = ["third time", "2nd time", "second time", "again and again", "baar baar", "already contacted", "multiple times"]
    if ans["priority"] == "P1" or any(w in low_t for w in repeat_words):
        if ans.get("needs_human") is not True:
            fixes.append("Set needs_human -> true")
        ans["needs_human"] = True
    else:
        ans["needs_human"] = bool(ans.get("needs_human", False))

    clean_json = json.dumps({
        "category": ans["category"],
        "priority": ans["priority"],
        "amount_paise": ans["amount_paise"],
        "txn_date": ans["txn_date"],
        "txn_id": ans["txn_id"],
        "channel": ans["channel"],
        "language": lang,
        "needs_human": ans["needs_human"],
    }, ensure_ascii=False)

    clean_train.append({
        "id": row_id,
        "messages": [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": ticket},
            {"role": "assistant", "content": clean_json}
        ]
    })
    if fixes:
        log_rows.append([row_id, "FIXED", "; ".join(fixes)])

save_jsonl("train_clean.jsonl", clean_train)
with open("cleaning_log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "action", "reason"])
    w.writerows(log_rows)

print(f"Saved {len(clean_train)} clean rows | Logged {len(log_rows)} actions.")
print("Sample CLEANED gold:", clean_train["messages"][-1]["content"])


Saved 1333 clean rows | Logged 281 actions.


TypeError: list indices must be integers or slices, not str

In [22]:
print("Verified sample cleaned label:", clean_train[-1]["messages"][-1]["content"])

results = [
    {"name": "exp1_raw", "exact_match": 95.0, "field_acc": 99.1, "gpu_gb": 8.38, "mins": 23.96},
    train_model("exp2_clean", clean_train, use_4bit=True, rank=16, lr=2e-4, epochs=3),
    train_model("exp3_16bit", clean_train, use_4bit=False, rank=16, lr=2e-4, epochs=3),
    train_model("exp4_rank32", clean_train, use_4bit=True, rank=32, lr=2e-4, epochs=3),
    train_model("exp5_4ep_lr1e4", clean_train, use_4bit=True, rank=16, lr=1e-4, epochs=4),
]

print("\n=== ALL EXPERIMENT RESULTS ===")
for r in results:
    print(r)


Verified sample cleaned label: {"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}
==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1333 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/1333 [00:00<?, ? examples/s]

... (30 outputs left out)

Step,Training Loss


Step,Training Loss
30,0.718629
60,0.030558
90,0.015731
120,0.008464
150,0.008766
180,0.003893
210,0.003392
240,0.003591
270,0.002172
300,0.001795


Unsloth: Restored added_tokens_decoder metadata in out_exp5_4ep_lr1e4/checkpoint-336/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ckpt_exp5_4ep_lr1e4/tokenizer_config.json.


DONE: {'name': 'exp5_4ep_lr1e4', 'exact_match': 92.0, 'field_acc': 98.4, 'gpu_gb': 10.63, 'mins': 29.51}

=== ALL EXPERIMENT RESULTS ===
{'name': 'exp1_raw', 'exact_match': 95.0, 'field_acc': 99.1, 'gpu_gb': 8.38, 'mins': 23.96}
{'name': 'exp2_clean', 'exact_match': 93.5, 'field_acc': 98.9, 'gpu_gb': 13.13, 'mins': 22.41}
{'name': 'exp3_16bit', 'exact_match': 93.0, 'field_acc': 98.6, 'gpu_gb': 13.2, 'mins': 18.84}
{'name': 'exp4_rank32', 'exact_match': 95.0, 'field_acc': 99.0, 'gpu_gb': 10.31, 'mins': 22.53}
{'name': 'exp5_4ep_lr1e4', 'exact_match': 92.0, 'field_acc': 98.4, 'gpu_gb': 10.63, 'mins': 29.51}


In [23]:
eligible = [r for r in results if r["name"] != "exp4_rank32"]
best = max(eligible, key=lambda x: (x["exact_match"], x["field_acc"]))["name"]
print("Best eligible model (<=50M params):", best)
print(subprocess.check_output(["python", "score.py", "--gold", "dev.jsonl", "--pred", f"pred_{best}.jsonl"]).decode())

preds = {p["id"]: p["output"] for p in load_jsonl(f"pred_{best}.jsonl")}
mistakes = []

for g in dev_rows:
    gold = json.loads(g["messages"][-1]["content"])
    user_msg = next(m["content"] for m in g["messages"] if m["role"] == "user")
    try:
        pred = json.loads(preds[g["id"]])
    except Exception:
        pred = None
    if pred != gold:
        mistakes.append((g["id"], user_msg, gold, preds[g["id"]]))

print(f"Total dev mistakes: {len(mistakes)} / {len(dev_rows)}")
for gid, txt, gold, pred in mistakes[:12]:
    print("-" * 50)
    print("ID    :", gid)
    print("TICKET:", txt[:300].replace("\n", " "))
    print("GOLD  :", gold)
    print("PRED  :", pred)


Best eligible model (<=50M params): exp1_raw
group                            n   valid   exact  fields
all                            200  100.0%   95.0%   99.1%
category=account_access         19  100.0%  100.0%  100.0%
category=fraud                  15  100.0%   93.3%   99.2%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   94.3%   98.8%
category=refund                 55  100.0%   89.1%   98.0%

per-field accuracy (all rows)
  category       100.0%
  priority        96.5%
  amount_paise   100.0%
  txn_date        98.5%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     97.5%

Total dev mistakes: 10 / 200
--------------------------------------------------
ID    : dv-00006
TICKET: Received: 2026-07-02 (Thu) Source: twitter  @NimbusPayCare The App shows a fraudulent payment m

In [24]:
import os
import shutil
from google.colab import files
from unsloth import FastLanguageModel

if os.path.exists("adapter"):
    shutil.rmtree("adapter")
shutil.copytree(f"ckpt_{best}", "adapter")


for fname in os.listdir("adapter"):
    if fname not in {"adapter_config.json", "adapter_model.safetensors", "README.md"}:
        os.remove(os.path.join("adapter", fname))

final_model, final_tok = FastLanguageModel.from_pretrained(
    model_name="adapter",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(final_model)

test_rows = load_jsonl("test_inputs.jsonl")
test_preds = predict_all(final_model, final_tok, test_rows, "predictions_test.jsonl", batch_size=4)
print("Saved test predictions:", len(test_preds))
print("First test prediction:", test_preds[0])

!zip -r final_submission.zip adapter train_clean.jsonl cleaning_log.csv predictions_test.jsonl
files.download("final_submission.zip")


==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Saved test predictions: 400
First test prediction: {'id': 'ts-00001', 'output': '{"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-01", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}'}
  adding: adapter/ (stored 0%)
  adding: adapter/README.md (deflated 65%)
  adding: adapter/adapter_model.safetensors (deflated 8%)
  adding: adapter/adapter_config.json (deflated 60%)
  adding: train_clean.jsonl (deflated 90%)
  adding: cleaning_log.csv (deflated 90%)
  adding: predictions_test.jsonl (deflated 93%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>